In [49]:
#This is a test to test the basic functions of the ACSPY package. This package lets you control the ACS controller using a wrapper
from acspy import acsc

#open the simulation of the controller (openCommDirect() will as default open a simulator. See further down how to open on ethernet)
hcomm = acsc.openCommDirect()

#We now enable the simulation axis 0. Axis are labeled as (0,1,6) (x,y,z) in the ASCII software
acsc.enable(hcomm,0)

#We now get the state of the motor, which will tell us what the stage is doing of the sepcific axis
acsc.getMotorState(hcomm, 0)
print(acsc.getMotorState(hcomm,0))

#ALWAYS REMEMBER TO CLOSE THE CURRENT CONNECTION!
#If you don't do this, the connection can be filled and needs to be cleared before more signals can be sent
#NB: If "error 135: All channels are busy" appears, then open SPIIplus User mode drive and disconnect each kernel.
acsc.closeComm(hcomm)

{'enabled': True, 'in position': True, 'moving': False, 'accelerating': False}


In [50]:
from acspy.control import Controller

#To start with we select the simulator to work with Controller function from the acspy package. 
controller = Controller(contype="simulator", n_axes=4)

#With this we can simply connect. The default connection is in this case correct for the current setup "Date: 2026-05-20"
controller.connect()

#We label and enable the x axis of the controller
axis0 = controller.axes[0]
axis0.enable()

#We then check to see if the axis is enabled (TRUE = enabled, FALSE = disabled)
print(axis0.enabled)

#Repeat this on the other 2 axis
axis1 = controller.axes[1]
axis1.enable()
print(axis1.enabled)
axis2 = controller.axes[2]
axis2.enable()
print(axis2.enabled)
axis0.disable()
print(axis0.enabled)

#As always remember to close the current connection
controller.disconnect()

True
True
True
False


In [51]:
from acspy import acsc
from acspy.control import Controller
import time

#Now we want to connect to the current stage controller. It has 3 axes and is connected by ethernet cable
controller = Controller(contype='ethernet',n_axes=3)

#The current setup (Date: 2026-05-20) works with the default IP and Port values.
controller.connect()

#We just name the communication for later control of the motors
comm =acsc.open_comm_ethernet_tcp()

#We define the axes and enable them. 
axis0 = controller.axes[0]
axis0.enable()

#The pause is enabled, as it takes a short while for the motors to enable. This was not required before as that was in a simulated environment
time.sleep(1)

#We now check the status of the axes. 
print(axis0.enabled)
print(acsc.getMotorState(comm,0))

#We do the same, but this time we disable the axis
time.sleep(1)
axis0.disable()
print(axis0.enabled)
print(acsc.getMotorState(comm,0))

#Remember to close the communications. Both controller and comm was opened, so clsoe both down. 
acsc.closeComm(comm)
controller.disconnect()

True
{'enabled': True, 'in position': True, 'moving': False, 'accelerating': False}
False
{'enabled': False, 'in position': True, 'moving': False, 'accelerating': False}


In [86]:
#Now we want to move the stage. 
#There are 2 different ways to move the stage. There is relative move and absolute move. 
#We also want to get the coordinates of the stage to make sure. 
#To start with we focus on axis 0. 
import sys
controller = Controller(contype='ethernet',n_axes=3)
controller.connect()

axis0 = controller.axes[0]
axis0.enable()
time.sleep(0.5)

#Since we are dealing with the stage controller we add a failsafe, to quit if the stages fail to enable
#(Note to self: Maybe replace with a while loop, to ensure the program stops if the stage disables for any reason during movement)
if axis0.enabled==True:
    print('The X axis is enabled')
else:
    sys.exit('The stage failed to enable!')

time.sleep(1)

#We then get the current position before we move to compare and see that the movement is succesful
print('rpos is before the move:',axis0.rpos)
time.sleep(0.5)

#now lets move it forward
#NB: The unit is mm.
axis0.ptpr(5)
time.sleep(2)

print('rpos is after 1st move:',axis0.rpos)

time.sleep(0.5)

#Now lets move it back a single step
axis0.ptpr(-5)
time.sleep(2)

print('rpos is after 2nd move:',axis0.rpos)
time.sleep(0.5)

#Always disable the axis at the end, and print the value for control. This is less important than disconnecting.
axis0.disable()
if axis0.enabled==False:
    print('The X axis has sucesfully been disabled')
else:
    print('The X axis failed to disable')

#Remember to always add disconnect at the end of a module to ensure no residual processes are run
controller.disconnect()

The X axis is enabled
rpos is before the move: -30.607792968750005
-0.005611687686037138
rpos is after 1st move: -25.607792968750005
False
rpos is after 2nd move: -30.607792968750005
True
The X axis has sucesfully been disabled


100.0